In [1]:
import pandas as pd
import numpy as np
import requests

url = ("https://archive-api.open-meteo.com/v1/archive"
       "?latitude=28.67&longitude=77.42"
       "&start_date=2015-01-01&end_date=2024-12-31"
       "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
       "precipitation_sum,wind_speed_10m_max"
       "&timezone=Asia%2FKolkata")

response = requests.get(url)
data = pd.DataFrame(response.json()['daily'])
data.head()

,time,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,2015-01-01,21.7,11.7,16.3,0.0,9.1
1,2015-01-02,16.6,14.0,15.0,4.2,9.0
2,2015-01-03,18.0,14.0,15.6,0.8,7.9
3,2015-01-04,19.1,10.6,14.6,0.0,13.5
4,2015-01-05,20.7,9.6,14.3,0.0,27.9


In [2]:
print("Rows and columns:", data.shape)
data.info()
print(data.isnull().sum())
data.describe()
data.to_csv("ghaziabad_weather_raw.csv", index=False)

Rows and columns: (3653, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3653 entries, 0 to 3652
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   time                 3653 non-null   object 
 1   temperature_2m_max   3653 non-null   float64
 2   temperature_2m_min   3653 non-null   float64
 3   temperature_2m_mean  3653 non-null   float64
 4   precipitation_sum    3653 non-null   float64
 5   wind_speed_10m_max   3653 non-null   float64
dtypes: float64(5), object(1)
memory usage: 171.4+ KB
time                   0
temperature_2m_max     0
temperature_2m_min     0
temperature_2m_mean    0
precipitation_sum      0
wind_speed_10m_max     0
dtype: int64


In [3]:
data = data.rename(columns={'time': 'date',
                            'temperature_2m_max': 'temp_max',
                            'temperature_2m_min': 'temp_min',
                            'temperature_2m_mean': 'temp_avg',
                            'precipitation_sum': 'rain_mm',
                            'wind_speed_10m_max': 'wind_max'})

data['date'] = pd.to_datetime(data['date'])
data['year'] = data['date'].dt.year
data['month'] = data['date'].dt.month
data['temp_range'] = data['temp_max'] - data['temp_min']
data['is_rainy'] = data['rain_mm'] > 0
data['temp_category'] = np.where(data['temp_avg'] > 30, 'Hot',
                         np.where(data['temp_avg'] > 18, 'Moderate', 'Cold'))
data.head()

,date,temp_max,temp_min,temp_avg,rain_mm,wind_max,year,month,temp_range,is_rainy,temp_category
0,2015-01-01,21.7,11.7,16.3,0.0,9.1,2015,1,10.0,False,Cold
1,2015-01-02,16.6,14.0,15.0,4.2,9.0,2015,1,2.6,True,Cold
2,2015-01-03,18.0,14.0,15.6,0.8,7.9,2015,1,4.0,True,Cold
3,2015-01-04,19.1,10.6,14.6,0.0,13.5,2015,1,8.5,False,Cold
4,2015-01-05,20.7,9.6,14.3,0.0,27.9,2015,1,11.1,False,Cold


In [4]:
yearly = data.groupby('year')['temp_avg'].mean()
print("Average temperature by year:")
print(yearly.round(2))
print("Change from first to last year:", round(yearly.iloc[-1] - yearly.iloc[0], 2), "°C")

Average temperature by year:
year
2015    24.26
2016    24.94
2017    24.46
2018    24.15
2019    24.00
2020    23.60
2021    24.11
2022    24.56
2023    23.96
2024    24.62
Name: temp_avg, dtype: float64
Change from first to last year: 0.37 °C


In [5]:
monthly = data.groupby('month')['temp_avg'].mean()
print("Average temperature by month:")
print(monthly.round(1))

Average temperature by month:
month
1     12.7
2     16.6
3     21.9
4     28.5
5     31.7
6     32.8
7     29.8
8     28.9
9     28.2
10    25.1
11    20.1
12    14.6
Name: temp_avg, dtype: float64


In [6]:
rain_month = data.groupby('month')['rain_mm'].sum()
rain_year = data.groupby('year')['rain_mm'].sum()
print("Total rainfall by month (mm):")
print(rain_month.round(1))
print("Total rainfall by year (mm):")
print(rain_year.round(1))
print("Wettest month:", rain_month.idxmax())
print("Wettest year:", rain_year.idxmax())

Total rainfall by month (mm):
month
1      316.7
2      251.5
3      320.8
4      108.4
5      309.5
6      614.7
7     2608.1
8     1742.9
9     1251.9
10     192.6
11      68.7
12      82.3
Name: rain_mm, dtype: float64
Total rainfall by year (mm):
year
2015     838.3
2016     801.3
2017     699.6
2018     673.4
2019     561.0
2020     588.1
2021     795.1
2022     763.0
2023     842.4
2024    1305.9
Name: rain_mm, dtype: float64
Wettest month: 7
Wettest year: 2024


In [7]:
range_month = data.groupby('month')['temp_range'].mean()
print("Average daily temperature range by month:")
print(range_month.round(1))
print("Biggest range in month:", range_month.idxmax())
print("Smallest range in month:", range_month.idxmin())
print("Average wind on rainy days:", round(data[data['is_rainy']]['wind_max'].mean(), 1))
print("Average wind on dry days:", round(data[~data['is_rainy']]['wind_max'].mean(), 1))

Average daily temperature range by month:
month
1     11.7
2     12.9
3     13.7
4     15.2
5     14.0
6     10.8
7      7.0
8      6.5
9      7.9
10    12.4
11    12.7
12    12.5
Name: temp_range, dtype: float64
Biggest range in month: 4
Smallest range in month: 8
Average wind on rainy days: 15.4
Average wind on dry days: 14.3


In [8]:
print(data[['temp_avg', 'temp_range', 'rain_mm', 'wind_max']].corr().round(2))

            temp_avg  temp_range  rain_mm  wind_max
temp_avg        1.00       -0.16     0.09      0.34
temp_range     -0.16        1.00    -0.41      0.01
rain_mm         0.09       -0.41     1.00      0.07
wind_max        0.34        0.01     0.07      1.00


In [9]:
print("Hottest month:", monthly.idxmax(), "->", round(monthly.max(), 1), "°C")
print("Coldest month:", monthly.idxmin(), "->", round(monthly.min(), 1), "°C")
print("Warmest year:", yearly.idxmax(), "->", round(yearly.max(), 2), "°C")
print("Coldest year:", yearly.idxmin(), "->", round(yearly.min(), 2), "°C")
print("Rainy days:", data['is_rainy'].sum(), "of", len(data))
print(data['temp_category'].value_counts())

Hottest month: 6 -> 32.8 °C
Coldest month: 1 -> 12.7 °C
Warmest year: 2016 -> 24.94 °C
Coldest year: 2020 -> 23.6 °C
Rainy days: 1293 of 3653
temp_category
Moderate    1970
Cold         890
Hot          793
Name: count, dtype: int64


In [10]:
from google.colab import files
data.to_csv("ghaziabad_weather_edited.csv", index=False)
files.download("ghaziabad_weather_edited.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>